# My Own Fast Frame Prediction Model

A lightweight residual temporal U-Net for Inter4K GOP-8 Y-channel frame prediction. It keeps the same input/output format as `model2.ipynb`: `(B, 7, 1, H, W) -> (B, 1, H, W)`.

Design goal: faster training than the ConvNeXt/Transformer reference while still targeting about 30 dB validation PSNR after training.

In [2]:
import os, math, random
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')  # Windows local workaround for duplicate OpenMP runtimes
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pathlib import Path

ON_KAGGLE = os.path.exists('/kaggle/working')

# for item in Path("/kaggle/working").iterdir():
#     if item.is_file():
#         item.unlink()
#     elif item.is_dir():
#         shutil.rmtree(item)

if ON_KAGGLE:
    DATASET_ROOT = '/kaggle/input/datasets/tonmoyk983/sevtone-4-qp-gop8/sevtone_4_QP_GOP8/Inter4K/RAW'
    CKPT_DIR = '/kaggle/working'
else:
    DATASET_ROOT = r'D:\\Dataset\\Inter4K\\60fps\\UHD\\Segments\\sevtone_4_QP_GOP8\\Inter4K\\RAW'
    CKPT_DIR = r'D:\\Dataset\\Inter4K\\60fps\\UHD\\Segments\\sevtone_4_QP_GOP8'

CKPT_DIR = Path(CKPT_DIR)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = CKPT_DIR / 'myown_fast_best_model.pth'
LAST_CKPT_PATH = CKPT_DIR / 'myown_fast_last_checkpoint.pth'

# Dataset / training config
T = 7
IMG_SIZE = 512
SUBSET_SIZE = 20000
VAL_FRACTION = 0.1
TRAIN_CROP_SIZE = 512   # set to 512 for full-image training; 256 is much faster

# Model / optimizer config
BASE_CHANNELS = 48       # increase to 64 for more PSNR, decrease to 32 for more speed
BATCH_SIZE = 4           # try 12/16 if your GPU has room
EPOCHS = 40
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
USE_AMP = True
USE_TQDM = False
BATCH_SUMMARY_EVERY = 100 
NUM_WORKERS = 2
RESUME_CKPT_PATH = "/kaggle/input/models/vaselinek983/check3/pytorch/3/1/myown_fast_last_checkpoint.pth"  # or str(LAST_CKPT_PATH)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

print(f'Running on: {"Kaggle" if ON_KAGGLE else "Local"}')
print(f'DATASET_ROOT : {DATASET_ROOT}')
print(f'Device       : {DEVICE}')
print(f'Best model   : {BEST_MODEL_PATH}')

Running on: Local
DATASET_ROOT : D:\\Dataset\\Inter4K\\60fps\\UHD\\Segments\\sevtone_4_QP_GOP8\\Inter4K\\RAW
Device       : cuda
Best model   : D:\Dataset\Inter4K\60fps\UHD\Segments\sevtone_4_QP_GOP8\myown_fast_best_model.pth


## Dataset

The loader follows `model2.ipynb`: input files are `sample_input_<id>.npy` with shape `(7, 512, 512)`, and target files are `sample_output_<id>.npy` with shape `(1, 512, 512)` or `(512, 512)`.

In [3]:
class Inter4KDataset(Dataset):
    def __init__(self, root, sample_ids, crop_size=None, random_crop=True):
        self.input_dir = Path(root) / 'Input'
        self.output_dir = Path(root) / 'Output'
        self.ids = list(sample_ids)
        self.crop_size = crop_size
        self.random_crop = random_crop

    def __len__(self):
        return len(self.ids)

    def _crop_pair(self, x, y):
        if self.crop_size is None:
            return x, y
        _, h, w = x.shape
        cs = min(self.crop_size, h, w)
        if self.random_crop:
            top = random.randint(0, h - cs)
            left = random.randint(0, w - cs)
        else:
            top = (h - cs) // 2
            left = (w - cs) // 2
        return x[:, top:top + cs, left:left + cs], y[:, top:top + cs, left:left + cs]

    def __getitem__(self, idx):
        sid = self.ids[idx]
        x = np.load(self.input_dir / f'sample_input_{sid}.npy')
        y = np.load(self.output_dir / f'sample_output_{sid}.npy')
        if y.ndim == 2:
            y = y[np.newaxis]

        x, y = self._crop_pair(x, y)
        x = np.ascontiguousarray(x.astype(np.float32) / 255.0)
        y = np.ascontiguousarray(y.astype(np.float32) / 255.0)

        x = torch.from_numpy(x).unsqueeze(1)  # (7, 1, H, W)
        y = torch.from_numpy(y)               # (1, H, W)
        return x, y


all_ids = list(range(1, SUBSET_SIZE + 1))
split_idx = int(len(all_ids) * (1 - VAL_FRACTION))
train_ids = all_ids[:split_idx]
val_ids = all_ids[split_idx:]

train_ds = Inter4KDataset(DATASET_ROOT, train_ids, crop_size=TRAIN_CROP_SIZE, random_crop=True)
val_ds = Inter4KDataset(DATASET_ROOT, val_ids, crop_size=None, random_crop=False)

loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda'),
    persistent_workers=(NUM_WORKERS > 0),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=max(1, BATCH_SIZE // 2), shuffle=False, **loader_kwargs)

print(f'Train: {len(train_ds)} samples | Val: {len(val_ds)} samples')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
x0, y0 = train_ds[0]
print(f'Sample x shape: {tuple(x0.shape)} | y shape: {tuple(y0.shape)}')

Train: 18000 samples | Val: 2000 samples
Train batches: 4500 | Val batches: 1000
Sample x shape: (7, 1, 512, 512) | y shape: (1, 512, 512)


## Fast Model

This model uses temporal channel stacking, first-order frame differences, a small two-level U-Net, depthwise-separable residual blocks, and residual prediction from the last input frame.

In [4]:
class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, groups=8):
        super().__init__()
        padding = kernel_size // 2
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride=stride, padding=padding, bias=False),
            nn.GroupNorm(min(groups, out_ch), out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class DSResBlock(nn.Module):
    def __init__(self, channels, expansion=2):
        super().__init__()
        hidden = channels * expansion
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 5, padding=2, groups=channels, bias=False),
            nn.GroupNorm(min(8, channels), channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
            nn.GroupNorm(min(8, channels), channels),
        )

    def forward(self, x):
        return x + self.net(x)


class FastTemporalUNet(nn.Module):
    def __init__(self, T=7, base_ch=48):
        super().__init__()
        self.T = T
        in_ch = T + (T - 1)  # raw frames plus temporal differences

        self.stem = nn.Sequential(
            ConvGNAct(in_ch, base_ch),
            DSResBlock(base_ch),
        )
        self.down1 = nn.Sequential(
            ConvGNAct(base_ch, base_ch * 2, stride=2),
            DSResBlock(base_ch * 2),
            DSResBlock(base_ch * 2),
        )
        self.down2 = nn.Sequential(
            ConvGNAct(base_ch * 2, base_ch * 3, stride=2),
            DSResBlock(base_ch * 3),
            DSResBlock(base_ch * 3),
            DSResBlock(base_ch * 3),
        )
        self.up1 = nn.Sequential(
            ConvGNAct(base_ch * 3 + base_ch * 2, base_ch * 2),
            DSResBlock(base_ch * 2),
        )
        self.up2 = nn.Sequential(
            ConvGNAct(base_ch * 2 + base_ch, base_ch),
            DSResBlock(base_ch),
        )
        self.head = nn.Conv2d(base_ch, 1, 3, padding=1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        # x: (B, T, 1, H, W)
        B, T_in, C, H, W = x.shape
        assert T_in == self.T and C == 1, f'Expected (B, {self.T}, 1, H, W), got {tuple(x.shape)}'
        frames = x.squeeze(2)                 # (B, T, H, W)
        diffs = frames[:, 1:] - frames[:, :-1] # (B, T-1, H, W)
        z = torch.cat([frames, diffs], dim=1)
        last = x[:, -1]

        s0 = self.stem(z)
        s1 = self.down1(s0)
        s2 = self.down2(s1)

        u1 = F.interpolate(s2, size=s1.shape[-2:], mode='bilinear', align_corners=False)
        u1 = self.up1(torch.cat([u1, s1], dim=1))
        u2 = F.interpolate(u1, size=s0.shape[-2:], mode='bilinear', align_corners=False)
        u2 = self.up2(torch.cat([u2, s0], dim=1))

        delta = self.head(u2)
        return torch.clamp(last + delta, 0.0, 1.0)


model = FastTemporalUNet(T=T, base_ch=BASE_CHANNELS)
model = model.to(DEVICE)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {total:,} total | {trainable:,} trainable')

Parameters: 843,889 total | 843,889 trainable


In [5]:
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, T, 1, TRAIN_CROP_SIZE, TRAIN_CROP_SIZE, device=DEVICE)
    out = model(dummy)
print(f'Input shape : {tuple(dummy.shape)}')
print(f'Output shape: {tuple(out.shape)}')
assert out.shape == (2, 1, TRAIN_CROP_SIZE, TRAIN_CROP_SIZE)
print('Shape verification passed')

Input shape : (2, 7, 1, 512, 512)
Output shape: (2, 1, 512, 512)
Shape verification passed


## Loss And Metrics

MSE is weighted highest because PSNR is the main target. A small Charbonnier term helps sharper residual learning without making training slow.

In [6]:
class FastPSNRLoss(nn.Module):
    def __init__(self, w_mse=0.85, w_charb=0.15, eps=1e-3):
        super().__init__()
        self.w_mse = w_mse
        self.w_charb = w_charb
        self.eps = eps

    def forward(self, pred, target):
        mse = F.mse_loss(pred, target)
        charb = torch.sqrt((pred - target) ** 2 + self.eps ** 2).mean()
        return self.w_mse * mse + self.w_charb * charb


def compute_psnr(pred, target, max_val=1.0):
    mse = F.mse_loss(pred, target).item()
    return float('inf') if mse < 1e-12 else 10.0 * math.log10(max_val ** 2 / mse)


def compute_ssim(pred, target, ws=11):
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    coords = torch.arange(ws, dtype=torch.float32, device=pred.device) - ws // 2
    g = torch.exp(-(coords ** 2) / (2 * 1.5 ** 2))
    g /= g.sum()
    k = (g.unsqueeze(0) * g.unsqueeze(1)).unsqueeze(0).unsqueeze(0).to(pred.dtype)
    p = ws // 2
    mu_x = F.conv2d(pred, k, padding=p)
    mu_y = F.conv2d(target, k, padding=p)
    sig_x = F.conv2d(pred * pred, k, padding=p) - mu_x * mu_x
    sig_y = F.conv2d(target * target, k, padding=p) - mu_y * mu_y
    sig_xy = F.conv2d(pred * target, k, padding=p) - mu_x * mu_y
    ssim = ((2 * mu_x * mu_y + C1) * (2 * sig_xy + C2)) / ((mu_x ** 2 + mu_y ** 2 + C1) * (sig_x + sig_y + C2))
    return ssim.mean().item()


criterion = FastPSNRLoss().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LEARNING_RATE * 0.05)
scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and DEVICE.type == 'cuda'))

print('Loss, optimizer, scheduler, PSNR, and SSIM are ready')

Loss, optimizer, scheduler, PSNR, and SSIM are ready


In [6]:
@torch.no_grad()
def evaluate_last_frame_baseline(loader, device, max_batches=20):
    run_psnr = 0.0
    n = 0
    for i, (x, y) in enumerate(loader):
        if i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        pred = x[:, -1]
        run_psnr += compute_psnr(pred, y)
        n += 1
    return run_psnr / max(1, n)


baseline_psnr = evaluate_last_frame_baseline(val_loader, DEVICE)
print(f'Last-input-frame baseline PSNR on first val batches: {baseline_psnr:.2f} dB')

RuntimeError: DataLoader worker (pid(s) 21208, 1880) exited unexpectedly

## Training

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, epoch):
    model.train()
    run_loss = 0.0
    run_psnr = 0.0
    run_ssim = 0.0
    window_loss = 0.0
    window_psnr = 0.0
    window_ssim = 0.0
    n = len(loader)
    pbar = tqdm(loader, desc=f'Epoch {epoch} [Train]', leave=False, disable=not USE_TQDM)
    optimizer.zero_grad(set_to_none=True)
    for i, (x, y) in enumerate(pbar):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(USE_AMP and device.type == 'cuda')):
            pred = model(x)
            loss = criterion(pred.float(), y.float())

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        pred_f = pred.detach().float()
        y_f = y.float()
        psnr = compute_psnr(pred_f, y_f)
        ssim = compute_ssim(pred_f, y_f)
        run_loss += loss.item()
        run_psnr += psnr
        run_ssim += ssim
        window_loss += loss.item()
        window_psnr += psnr
        window_ssim += ssim
        if USE_TQDM:
            pbar.set_postfix(loss=f'{run_loss / (i + 1):.4f}', psnr=f'{psnr:.2f}dB', ssim=f'{ssim:.4f}')
        should_report = ((i + 1) % BATCH_SUMMARY_EVERY == 0) or ((i + 1) == n)
        if should_report:
            batch_count = BATCH_SUMMARY_EVERY if ((i + 1) % BATCH_SUMMARY_EVERY == 0) else ((i + 1) % BATCH_SUMMARY_EVERY or BATCH_SUMMARY_EVERY)
            start_batch = i + 2 - batch_count
            end_batch = i + 1
            print(
                f'[Train][Epoch {epoch}] Batches {start_batch}-{end_batch}/{n} | '
                f'avg loss={window_loss / batch_count:.4f} | '
                f'avg PSNR={window_psnr / batch_count:.2f}dB | '
                f'avg SSIM={window_ssim / batch_count:.4f}'
            )
            window_loss = 0.0
            window_psnr = 0.0
            window_ssim = 0.0
    return {'loss': run_loss / n, 'psnr': run_psnr / n, 'ssim': run_ssim / n}


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    run_loss = 0.0
    run_psnr = 0.0
    run_ssim = 0.0
    window_loss = 0.0
    window_psnr = 0.0
    window_ssim = 0.0
    n = len(loader)
    pbar = tqdm(loader, desc='Validation', leave=False, disable=not USE_TQDM)
    for i, (x, y) in enumerate(pbar):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(USE_AMP and device.type == 'cuda')):
            pred = model(x)
        pred = pred.float()
        y = y.float()
        loss = criterion(pred, y).item()
        psnr = compute_psnr(pred, y)
        ssim = compute_ssim(pred, y)
        run_loss += loss
        run_psnr += psnr
        run_ssim += ssim
        window_loss += loss
        window_psnr += psnr
        window_ssim += ssim
        if USE_TQDM:
            pbar.set_postfix(loss=f'{loss:.4f}', psnr=f'{psnr:.2f}dB', ssim=f'{ssim:.4f}')
        should_report = ((i + 1) % BATCH_SUMMARY_EVERY == 0) or ((i + 1) == n)
        if should_report:
            batch_count = BATCH_SUMMARY_EVERY if ((i + 1) % BATCH_SUMMARY_EVERY == 0) else ((i + 1) % BATCH_SUMMARY_EVERY or BATCH_SUMMARY_EVERY)
            start_batch = i + 2 - batch_count
            end_batch = i + 1
            print(
                f'[Val] Batches {start_batch}-{end_batch}/{n} | '
                f'avg loss={window_loss / batch_count:.4f} | '
                f'avg PSNR={window_psnr / batch_count:.2f}dB | '
                f'avg SSIM={window_ssim / batch_count:.4f}'
            )
            window_loss = 0.0
            window_psnr = 0.0
            window_ssim = 0.0
    return {'loss': run_loss / n, 'psnr': run_psnr / n, 'ssim': run_ssim / n}


print('train_one_epoch and validate defined')

In [ ]:
start_epoch = 1
best_psnr = 0.0
history = {'train_loss': [], 'train_psnr': [], 'val_loss': [], 'val_psnr': [], 'val_ssim': []}

if RESUME_CKPT_PATH and os.path.exists(RESUME_CKPT_PATH):
    print(f'Loading checkpoint: {RESUME_CKPT_PATH}')
    ckpt = torch.load(RESUME_CKPT_PATH, map_location=DEVICE)
    if isinstance(ckpt, dict) and 'model' in ckpt:
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        if 'scheduler' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt.get('epoch', 0) + 1
        best_psnr = ckpt.get('best_psnr', 0.0)
        history = ckpt.get('history', history)
    else:
        model.load_state_dict(ckpt)
    print(f'Resumed from epoch {start_epoch} | best PSNR: {best_psnr:.2f} dB')
else:
    print('No checkpoint loaded - training from scratch')

In [ ]:
for epoch in range(start_epoch, start_epoch + EPOCHS):
    print(f'\n{"=" * 55}')
    print(f'Epoch {epoch}  (run epochs {start_epoch} - {start_epoch + EPOCHS - 1})')
    print(f'{"=" * 55}')

    tr = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, epoch)
    va = validate(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history['train_loss'].append(tr['loss'])
    history['train_psnr'].append(tr['psnr'])
    history['val_loss'].append(va['loss'])
    history['val_psnr'].append(va['psnr'])
    history['val_ssim'].append(va['ssim'])

    print(f"Train loss={tr['loss']:.4f}  PSNR={tr['psnr']:.2f}dB")
    print(f"Val   loss={va['loss']:.4f}  PSNR={va['psnr']:.2f}dB  SSIM={va['ssim']:.4f}")

    if va['psnr'] > best_psnr:
        best_psnr = va['psnr']
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f'[BEST] PSNR={best_psnr:.2f}dB -> {BEST_MODEL_PATH}')

    torch.save({
        'epoch': epoch,
        'best_psnr': best_psnr,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'history': history,
        'config': {
            'T': T,
            'base_channels': BASE_CHANNELS,
            'train_crop_size': TRAIN_CROP_SIZE,
            'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
        },
    }, LAST_CKPT_PATH)

print(f'\nRun complete. Best Val PSNR: {best_psnr:.2f} dB')
print(f'Inference weights: {BEST_MODEL_PATH}')
print(f'Resume checkpoint : {LAST_CKPT_PATH}')

## Curves And Demo

In [ ]:
epochs_seen = range(1, len(history['val_psnr']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_seen, history['train_loss'], label='Train', marker='o')
axes[0].plot(epochs_seen, history['val_loss'], label='Val', marker='s')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_seen, history['train_psnr'], label='Train', marker='o')
axes[1].plot(epochs_seen, history['val_psnr'], label='Val', marker='s')
axes[1].set_title('PSNR (dB)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(epochs_seen, history['val_ssim'], color='purple', marker='o')
axes[2].set_title('Val SSIM')
axes[2].set_xlabel('Epoch')
axes[2].grid(True)

plt.suptitle('FastTemporalUNet Training Curves')
plt.tight_layout()
curves_path = CKPT_DIR / 'myown_fast_training_curves.png'
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
print(f'Curves saved -> {curves_path}')
plt.show()

In [ ]:
def load_single_sample(sid):
    ds = Inter4KDataset(DATASET_ROOT, [sid], crop_size=None, random_crop=False)
    x, y = ds[0]
    return x.unsqueeze(0), y.unsqueeze(0)


if BEST_MODEL_PATH.exists():
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model.eval()

DEMO_ID = 1
x_d, y_d = load_single_sample(DEMO_ID)
with torch.no_grad():
    y_p = model(x_d.to(DEVICE)).cpu()

psnr_v = compute_psnr(y_p, y_d)
ssim_v = compute_ssim(y_p.to(DEVICE), y_d.to(DEVICE))

fig, axes = plt.subplots(1, T + 2, figsize=(3 * (T + 2), 3.5))
for t in range(T):
    axes[t].imshow(x_d[0, t, 0].numpy(), cmap='gray', vmin=0, vmax=1)
    axes[t].set_title(f'Input t={t}', fontsize=8)
    axes[t].axis('off')

axes[T].imshow(y_d[0, 0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[T].set_title('Target', fontsize=8)
axes[T].axis('off')

axes[T + 1].imshow(y_p[0, 0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[T + 1].set_title(f'Predicted\nPSNR={psnr_v:.2f}dB\nSSIM={ssim_v:.4f}', fontsize=8)
axes[T + 1].axis('off')

plt.suptitle(f'Sample {DEMO_ID} - FastTemporalUNet Output')
plt.tight_layout()
plt.show()

In [7]:
import csv
import re

TESTING_CLASSD2_ROOT = Path(DATASET_ROOT).parent.parent / 'Testing_ClassA'
ORIGINAL_CLASSD_ROOT = Path(DATASET_ROOT).parent.parent / 'ClassA'
EVAL_MODEL_PATH = CKPT_DIR / 'myown_fast_best_model_final.pth'
if not EVAL_MODEL_PATH.exists():
    EVAL_MODEL_PATH = BEST_MODEL_PATH

RESIZE_INFERENCE_TO_512 = True
INFERENCE_SIZE = 512
GOP_SIZE = 8
OUTPUT_ROOT = CKPT_DIR / 'frame_prediction_csvs/ClassA'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

base_model = model.module if isinstance(model, nn.DataParallel) else model
base_model.load_state_dict(torch.load(EVAL_MODEL_PATH, map_location=DEVICE))
base_model.eval()
print(f'Loaded checkpoint: {EVAL_MODEL_PATH}')
print(f'Testing root      : {TESTING_CLASSD2_ROOT}')
print(f'Original root     : {ORIGINAL_CLASSD_ROOT}')
print(f'CSV output root   : {OUTPUT_ROOT}')


def read_yuv420_y_frames(yuv_path, width, height):
    frame_size = width * height * 3 // 2
    raw = np.fromfile(yuv_path, dtype=np.uint8)
    if raw.size % frame_size != 0:
        raise ValueError(f'File size mismatch for {yuv_path} (expected multiples of {frame_size} bytes per frame)')
    num_frames = raw.size // frame_size
    y_size = width * height
    frames = raw.reshape(num_frames, frame_size)
    y_frames = frames[:, :y_size].reshape(num_frames, height, width)
    return y_frames


def parse_resolution(folder_name):
    match = re.search(r'(\d+)x(\d+)', folder_name)
    if not match:
        raise ValueError(f'Could not parse resolution from {folder_name}')
    return int(match.group(1)), int(match.group(2))


def find_resolution_from_path(path):
    for part in [path.name, *path.parts]:
        match = re.search(r'(\d+)x(\d+)', part)
        if match:
            return int(match.group(1)), int(match.group(2))
    raise ValueError(f'Could not parse resolution from path {path}')


def resolve_original_yuv_path(recon_yuv_path):
    video_folder = recon_yuv_path.parent.parent.name
    if not ORIGINAL_CLASSD_ROOT.exists():
        raise FileNotFoundError(f'Original Class D root not found: {ORIGINAL_CLASSD_ROOT}')

    candidates = [p for p in ORIGINAL_CLASSD_ROOT.rglob('*.yuv') if p != recon_yuv_path]
    if not candidates:
        raise FileNotFoundError(f'No original YUV files found under {ORIGINAL_CLASSD_ROOT}')

    def score(path):
        path_text = str(path).lower()
        score_value = 0
        if path.parent.name == video_folder:
            score_value += 10
        if path.stem == video_folder:
            score_value += 8
        if video_folder.lower() in path.stem.lower():
            score_value += 6
        if 'qp_' not in path.name.lower():
            score_value += 2
        if 'testing_classd2' not in path_text:
            score_value += 1
        return score_value

    ranked = sorted(((score(p), p) for p in candidates), key=lambda item: (item[0], len(item[1].parts)), reverse=True)
    best_score, best_path = ranked[0]
    if best_score <= 0:
        raise FileNotFoundError(f'Could not match an original YUV for {recon_yuv_path}')
    return best_path


def resize_frame_stack(frames, size=INFERENCE_SIZE):
    if not RESIZE_INFERENCE_TO_512:
        return frames
    tensor = torch.from_numpy(frames).unsqueeze(1).float()
    tensor = F.interpolate(tensor, size=(size, size), mode='bilinear', align_corners=False)
    return tensor.squeeze(1).cpu().numpy()


def load_frame_stack(yuv_path):
    width, height = find_resolution_from_path(yuv_path)
    frames = read_yuv420_y_frames(yuv_path, width, height).astype(np.float32) / 255.0
    return resize_frame_stack(frames)


def frame_tensor(frames, frame_index):
    return torch.from_numpy(frames[frame_index - 1:frame_index]).unsqueeze(1).to(DEVICE)


def parse_encode_log(log_path):
    frame_pattern = re.compile(r'POC\s+(\d+).*?\[Y\s+([0-9.]+)\s+dB', re.IGNORECASE)
    summary_pattern = re.compile(r'^\s*(\d+)\s+a\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s*$', re.IGNORECASE)
    frame_rows = []
    summary = {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as handle:
        for line in handle:
            frame_match = frame_pattern.search(line)
            if frame_match:
                poc = int(frame_match.group(1))
                frame_rows.append({
                    'frame_index': poc + 1,
                    'source': 'vvc',
                    'y_psnr': float(frame_match.group(2)),
                    'ssim': '',
                    'note': 'encode_log',
                })
                continue
            summary_match = summary_pattern.match(line)
            if summary_match:
                summary = {
                    'frame_count': int(summary_match.group(1)),
                    'bitrate': float(summary_match.group(2)),
                    'avg_y_psnr': float(summary_match.group(3)),
                    'avg_u_psnr': float(summary_match.group(4)),
                    'avg_v_psnr': float(summary_match.group(5)),
                    'avg_yuv_psnr': float(summary_match.group(6)),
                }
    if not frame_rows:
        raise ValueError(f'No frame PSNR entries found in {log_path}')
    return frame_rows, summary


def model_target_frames(max_frame_index):
    return list(range(GOP_SIZE, max_frame_index + 1, GOP_SIZE))


@torch.no_grad()
def build_vvc_rows(frame_rows, recon_frames, orig_frames):
    max_frame_index = max(row['frame_index'] for row in frame_rows)
    if recon_frames.shape[0] < max_frame_index:
        raise ValueError(f'Need at least {max_frame_index} reconstructed frames, found {recon_frames.shape[0]}')
    if orig_frames.shape[0] < max_frame_index:
        raise ValueError(f'Need at least {max_frame_index} original frames, found {orig_frames.shape[0]}')

    vvc_rows = []
    for row in frame_rows:
        frame_index = row['frame_index']
        recon = frame_tensor(recon_frames, frame_index)
        orig = frame_tensor(orig_frames, frame_index)
        vvc_rows.append({
            'row_type': 'frame',
            'frame_index': frame_index,
            'source': 'vvc',
            'y_psnr': row['y_psnr'],
            'ssim': compute_ssim(recon, orig),
            'note': 'encode_log_psnr_computed_ssim',
        })
    return vvc_rows


@torch.no_grad()
def predict_model_rows(recon_frames, orig_frames, target_frames):
    if not target_frames:
        raise ValueError('No GOP target frames were found for model prediction')
    if recon_frames.shape[0] < max(target_frames):
        raise ValueError(f'Need at least {max(target_frames)} reconstructed frames, found {recon_frames.shape[0]}')
    if orig_frames.shape[0] < max(target_frames):
        raise ValueError(f'Need at least {max(target_frames)} original frames, found {orig_frames.shape[0]}')

    rows = {}
    for frame_index in target_frames:
        start = frame_index - GOP_SIZE
        end = frame_index - 1
        if start < 0:
            raise ValueError(f'Frame {frame_index} does not have {GOP_SIZE - 1} previous input frames')
        x_np = recon_frames[start:end]
        y_np = orig_frames[frame_index - 1:frame_index]
        x = torch.from_numpy(x_np).unsqueeze(0).unsqueeze(2).to(DEVICE)
        y = torch.from_numpy(y_np).unsqueeze(0).to(DEVICE)
        pred = base_model(x).float()
        rows[frame_index] = {
            'row_type': 'frame',
            'frame_index': frame_index,
            'source': 'model',
            'y_psnr': compute_psnr(pred.cpu(), y.cpu()),
            'ssim': compute_ssim(pred.cpu(), y.cpu()),
            'note': f'predicted from frames {frame_index - (GOP_SIZE - 1)}-{frame_index - 1}',
        }
    return rows


def build_mixed_rows(vvc_rows, model_rows):
    mixed_rows = []
    for vvc_row in sorted(vvc_rows, key=lambda row: row['frame_index']):
        frame_index = vvc_row['frame_index']
        if frame_index in model_rows:
            mixed_rows.append(model_rows[frame_index])
        else:
            mixed_rows.append(vvc_row)
    return mixed_rows


def mean_metric(rows, metric):
    values = [row[metric] for row in rows if row.get(metric) != '']
    return float(np.mean(values)) if values else ''


def summarize_report_rows(rows, report_name, log_summary, target_frames):
    vvc_rows = [row for row in rows if row['source'] == 'vvc']
    model_rows = [row for row in rows if row['source'] == 'model']
    summary_rows = [
        {
            'row_type': 'summary',
            'video_id': '',
            'qp': '',
            'frame_index': 'all',
            'source': f'{report_name}_mean',
            'y_psnr': mean_metric(rows, 'y_psnr'),
            'ssim': mean_metric(rows, 'ssim'),
            'note': f"encode_log_reported_yuv_psnr={log_summary.get('avg_yuv_psnr', '')}",
        },
    ]
    if vvc_rows:
        summary_rows.append({
            'row_type': 'summary',
            'video_id': '',
            'qp': '',
            'frame_index': 'vvc',
            'source': 'vvc_mean_in_report',
            'y_psnr': mean_metric(vvc_rows, 'y_psnr'),
            'ssim': mean_metric(vvc_rows, 'ssim'),
            'note': 'vvc rows included in this CSV',
        })
    if model_rows:
        summary_rows.append({
            'row_type': 'summary',
            'video_id': '',
            'qp': '',
            'frame_index': 'model',
            'source': 'model_mean_in_report',
            'y_psnr': mean_metric(model_rows, 'y_psnr'),
            'ssim': mean_metric(model_rows, 'ssim'),
            'note': f'model frames {target_frames}',
        })
    return summary_rows


def write_report_csv(csv_path, rows, summary_rows, video_id, qp):
    fieldnames = [
        'row_type',
        'video_id',
        'qp',
        'frame_index',
        'source',
        'y_psnr',
        'ssim',
        'note',
    ]
    with open(csv_path, 'w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows + summary_rows:
            output_row = {
                'row_type': row.get('row_type', 'frame'),
                'video_id': video_id,
                'qp': qp,
                'frame_index': row.get('frame_index', ''),
                'source': row.get('source', ''),
                'y_psnr': row.get('y_psnr', ''),
                'ssim': row.get('ssim', ''),
                'note': row.get('note', ''),
            }
            writer.writerow(output_row)


def process_qp_folder(qp_dir):
    log_path = qp_dir / 'encode.log'
    if not log_path.exists():
        raise FileNotFoundError(f'Missing encode.log in {qp_dir}')

    frame_rows, log_summary = parse_encode_log(log_path)
    recon_yuv_paths = sorted(qp_dir.glob('*.yuv'))
    if not recon_yuv_paths:
        raise FileNotFoundError(f'No reconstructed YUV files found in {qp_dir}')

    video_id = qp_dir.parent.name
    qp = qp_dir.name
    written_csvs = []

    for recon_yuv_path in recon_yuv_paths:
        orig_yuv_path = resolve_original_yuv_path(recon_yuv_path)
        recon_frames = load_frame_stack(recon_yuv_path)
        orig_frames = load_frame_stack(orig_yuv_path)
        max_frame_index = max(row['frame_index'] for row in frame_rows)
        target_frames = model_target_frames(max_frame_index)

        vvc_rows = build_vvc_rows(frame_rows, recon_frames, orig_frames)
        model_rows = predict_model_rows(recon_frames, orig_frames, target_frames)
        mixed_rows = build_mixed_rows(vvc_rows, model_rows)

        vvc_summary_rows = summarize_report_rows(vvc_rows, 'vvc_all', log_summary, target_frames)
        mixed_summary_rows = summarize_report_rows(mixed_rows, 'vvc_1_7_model_8', log_summary, target_frames)

        out_dir = OUTPUT_ROOT / video_id / qp
        out_dir.mkdir(parents=True, exist_ok=True)
        vvc_csv_path = out_dir / f'{recon_yuv_path.stem}_vvc_all.csv'
        mixed_csv_path = out_dir / f'{recon_yuv_path.stem}_vvc_1_7_model_8.csv'

        write_report_csv(vvc_csv_path, vvc_rows, vvc_summary_rows, video_id, qp)
        write_report_csv(mixed_csv_path, mixed_rows, mixed_summary_rows, video_id, qp)
        written_csvs.extend([vvc_csv_path, mixed_csv_path])

        print(f'Wrote {vvc_csv_path}')
        print(f'Wrote {mixed_csv_path}')
        print(f"  VVC all Y-PSNR mean       : {mean_metric(vvc_rows, 'y_psnr'):.4f}")
        print(f"  VVC all SSIM mean         : {mean_metric(vvc_rows, 'ssim'):.4f}")
        print(f"  VVC reported YUV-PSNR     : {log_summary.get('avg_yuv_psnr', float('nan')):.4f}")
        print(f"  Mixed Y-PSNR mean         : {mean_metric(mixed_rows, 'y_psnr'):.4f}")
        print(f"  Mixed SSIM mean           : {mean_metric(mixed_rows, 'ssim'):.4f}")
        print(f"  Model target mean Y-PSNR  : {mean_metric(list(model_rows.values()), 'y_psnr'):.4f}")
        print(f"  Model target mean SSIM    : {mean_metric(list(model_rows.values()), 'ssim'):.4f}")

    return written_csvs


results = []
for qp_dir in sorted(TESTING_CLASSD2_ROOT.glob('*/*')):
    if qp_dir.is_dir() and qp_dir.name.startswith('QP_'):
        results.extend(process_qp_folder(qp_dir))

if not results:
    raise RuntimeError(f'No QP folders found under {TESTING_CLASSD2_ROOT}')

print(f'\nFinished CSV export. Wrote {len(results)} CSV files.')

Loaded checkpoint: D:\Dataset\Inter4K\60fps\UHD\Segments\sevtone_4_QP_GOP8\myown_fast_best_model_final.pth
Testing root      : D:\Dataset\Inter4K\60fps\UHD\Segments\sevtone_4_QP_GOP8\Testing_ClassA
Original root     : D:\Dataset\Inter4K\60fps\UHD\Segments\sevtone_4_QP_GOP8\ClassA
CSV output root   : D:\Dataset\Inter4K\60fps\UHD\Segments\sevtone_4_QP_GOP8\frame_prediction_csvs\ClassA
Wrote D:\Dataset\Inter4K\60fps\UHD\Segments\sevtone_4_QP_GOP8\frame_prediction_csvs\ClassA\Traffic_2560x1600_30\QP_37\C_Traffic_2560x1600_30_QP_37_F_1_to_32_vvc_all.csv
Wrote D:\Dataset\Inter4K\60fps\UHD\Segments\sevtone_4_QP_GOP8\frame_prediction_csvs\ClassA\Traffic_2560x1600_30\QP_37\C_Traffic_2560x1600_30_QP_37_F_1_to_32_vvc_1_7_model_8.csv
  VVC all Y-PSNR mean       : 34.5292
  VVC all SSIM mean         : 0.9474
  VVC reported YUV-PSNR     : 35.3941
  Mixed Y-PSNR mean         : 34.0191
  Mixed SSIM mean           : 0.9439
  Model target mean Y-PSNR  : 30.3740
  Model target mean SSIM    : 0.9185
Wrote

In [8]:
import csv
import re
from pathlib import Path

# Bitrate formulas copied from Archive/compute_bitrate.py:
#   reported_bitrate = (total_bits / total_frames) * frame_rate / 1000
#   new_bitrate      = (new_total_bits / total_frames) * frame_rate / 1000

BITRATE_TESTING_ROOT = Path(DATASET_ROOT).parent.parent / 'Testing_ClassA'
BITRATE_OUTPUT_ROOT = CKPT_DIR / 'frame_prediction_csvs'
BITRATE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
BITRATE_SUMMARY_CSV = BITRATE_OUTPUT_ROOT / 'bitrate_summary4.csv'

EXCLUDED_FRAME_INDICES = [8, 16, 24, 32]
FORCED_FRAME_RATE = None  # set to 50 or 60 to force fps; None derives it from the VTM summary

BITRATE_FRAME_RE = re.compile(r'POC\s+(\d+)\s+.*?\)\s+(\d+)\s+bits')
BITRATE_SUMMARY_RE = re.compile(
    r'^\s*(\d+)\s+a\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s*$',
    re.MULTILINE,
)


def parse_bitrate_log(log_path):
    text = Path(log_path).read_text(encoding='utf-8', errors='ignore')
    frames = {int(poc): int(bits) for poc, bits in BITRATE_FRAME_RE.findall(text)}
    if not frames:
        raise ValueError(f"No per-frame 'POC ... bits' lines found in {log_path}")

    summary_match = BITRATE_SUMMARY_RE.search(text)
    if not summary_match:
        raise ValueError(f'Could not find VTM summary line in {log_path}')

    return {
        'frames': frames,
        'total_frames': int(summary_match.group(1)),
        'reported_bitrate_kbps': float(summary_match.group(2)),
    }


def derive_frame_rate(total_bits, total_frames, reported_bitrate_kbps):
    return reported_bitrate_kbps * 1000.0 * total_frames / total_bits


def calculate_bitrate_rows(log_path, excluded_frame_indices=EXCLUDED_FRAME_INDICES):
    parsed = parse_bitrate_log(log_path)
    frames = parsed['frames']
    total_frames = parsed['total_frames']
    reported_bitrate = parsed['reported_bitrate_kbps']
    total_bits = sum(frames.values())
    frame_rate = FORCED_FRAME_RATE if FORCED_FRAME_RATE else derive_frame_rate(total_bits, total_frames, reported_bitrate)

    excluded_pocs = sorted(poc for poc in frames if poc + 1 in excluded_frame_indices)
    excluded_bits = sum(frames[poc] for poc in excluded_pocs)
    kept_bits = total_bits - excluded_bits

    all_frames_bitrate = (total_bits / total_frames) * frame_rate / 1000.0
    except_selected_bitrate = (kept_bits / total_frames) * frame_rate / 1000.0
    reduction_pct = 100.0 * (all_frames_bitrate - except_selected_bitrate) / all_frames_bitrate

    video_id = log_path.parent.parent.name
    qp = log_path.parent.name
    return [
        {
            'video_id': video_id,
            'qp': qp,
            'case': 'all_frames',
            'frame_indices': 'all',
            'pocs_removed': '',
            'bits_removed': 0,
            'total_bits_used': total_bits,
            'total_encoded_bits': total_bits,
            'total_frames_denominator': total_frames,
            'parsed_frame_count': len(frames),
            'derived_frame_rate': frame_rate,
            'bitrate_kbps': all_frames_bitrate,
            'reported_bitrate_kbps': reported_bitrate,
            'bitrate_reduction_pct': 0.0,
            'log_path': str(log_path),
        },
        {
            'video_id': video_id,
            'qp': qp,
            'case': 'all_except_8_16_24_32',
            'frame_indices': ','.join(map(str, excluded_frame_indices)),
            'pocs_removed': ','.join(map(str, excluded_pocs)),
            'bits_removed': excluded_bits,
            'total_bits_used': kept_bits,
            'total_encoded_bits': total_bits,
            'total_frames_denominator': total_frames,
            'parsed_frame_count': len(frames),
            'derived_frame_rate': frame_rate,
            'bitrate_kbps': except_selected_bitrate,
            'reported_bitrate_kbps': reported_bitrate,
            'bitrate_reduction_pct': reduction_pct,
            'log_path': str(log_path),
        },
    ]


bitrate_rows = []
for log_path in sorted(BITRATE_TESTING_ROOT.glob('*/*/encode.log')):
    bitrate_rows.extend(calculate_bitrate_rows(log_path))

if not bitrate_rows:
    raise RuntimeError(f'No encode.log files found under {BITRATE_TESTING_ROOT}')

bitrate_fieldnames = list(bitrate_rows[0].keys())
with open(BITRATE_SUMMARY_CSV, 'w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=bitrate_fieldnames)
    writer.writeheader()
    writer.writerows(bitrate_rows)

print(f'Wrote bitrate summary: {BITRATE_SUMMARY_CSV}')
for row in bitrate_rows:
    print(
        f"{row['video_id']} {row['qp']} {row['case']}: "
        f"{row['bitrate_kbps']:.4f} kbps, "
        f"removed {row['bits_removed']} bits, "
        f"reduction {row['bitrate_reduction_pct']:.2f}%"
    )

Wrote bitrate summary: D:\Dataset\Inter4K\60fps\UHD\Segments\sevtone_4_QP_GOP8\frame_prediction_csvs\bitrate_summary4.csv
Traffic_2560x1600_30 QP_37 all_frames: 2898.3675 kbps, removed 0 bits, reduction 0.00%
Traffic_2560x1600_30 QP_37 all_except_8_16_24_32: 2753.8350 kbps, removed 154168 bits, reduction 4.99%
Traffic_2560x1600_30 QP_42 all_frames: 1482.8325 kbps, removed 0 bits, reduction 0.00%
Traffic_2560x1600_30 QP_42 all_except_8_16_24_32: 1419.5850 kbps, removed 67464 bits, reduction 4.27%
Traffic_2560x1600_30 QP_47 all_frames: 715.6350 kbps, removed 0 bits, reduction 0.00%
Traffic_2560x1600_30 QP_47 all_except_8_16_24_32: 686.9475 kbps, removed 30600 bits, reduction 4.01%
Traffic_2560x1600_30 QP_51 all_frames: 376.1475 kbps, removed 0 bits, reduction 0.00%
Traffic_2560x1600_30 QP_51 all_except_8_16_24_32: 362.3700 kbps, removed 14696 bits, reduction 3.66%
